# Custom Classes Comparasion Class Test

In [ ]:
import torch
from senpi.custom_classes.classes import *

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Example usage
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = torch.tensor([2.0, 2.0, 1.0], requires_grad=True)
z = torch.tensor([5.0, 5.0, 10.0])
min_val = torch.tensor(0.0)
max_val = torch.tensor(10.0)

output = torch.max(CustomComparison.apply(x, y, '==', min_val, max_val), z)
output.backward(torch.ones_like(output))

print("Output:", output)
print("Gradients - x:", x.grad)
print("Gradients - y:", y.grad)


In [ ]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def approx_equal(a, b, k=10):
    return sigmoid(k * abs(a - b))

# Test the function
a1, b1 = -10.0, 1.0
a2, b2 = 1, 1.0

result1 = 2*(1-approx_equal(a1, b1))  # Should be close to 1
result2 = 2*approx_equal(a2, b2)-1  # Should be closer to 0

print(f"Approximate equality of {a1} and {b1}: {result1}")  # Output: close to 1
print(f"Approximate equality of {a2} and {b2}: {result2}")  # Output: close to 0

# Simulation Demo

In [ ]:
import senpi
from senpi import *
from senpi.sim.simulator import EventSimulator
from senpi.sim.params import make_params
import torch
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# First, generate user params and check if accurate
params = make_params()
params["shot_noise"] = 1
params["sensor_noise"] = 1
print(params)

# Next, create instance of event simulator class and check some initialized quantities
es = EventSimulator(params=params)
print(es.params)  # should match above
print(es.hot_map)  # empty upon initialization
print(es.noise_map)
print(es.pos_map)

# Now, test initialization of internal grids of event camera and recheck initialized quantities
es._generate_internal_properties(im_size = [50, 200])
print(es.hot_map)  # empty upon initialization
print(es.noise_map)
print(es.pos_map)

In [ ]:
from tqdm import tqdm
from notebook_video_writer import VideoWriter

# Lets generate a dummy object to test the forward model - a spatiotemporal sinusoid
vid_size = [250, 50, 200]  # declared in [t, x, y]
v1 = torch.ones(size=vid_size[1:]) * 0.02  # explicitly declare a low uniform starting reference voltage (else defined in params)
stack = torch.zeros(size=vid_size, dtype=torch.float)  # preallocate
peak_photons = params["well_cap"]  # peak of signal is well cap of sensor
fx = 1/50  # spatial frequency in units of 1/pixels
ft = 1/200  # temporal frequency in units of 1/dt

# set up appropriate grids for video object
xx = torch.arange(0, vid_size[2], dtype=torch.float32)
xx = xx.repeat(vid_size[1], 1)

# write a video to the notebook to visualize result!
with VideoWriter(filename="es_intensity.mp4", fps=30) as vw:
    for i in tqdm(range(vid_size[0])):

        # # for visualization
        frame = (torch.sin(2*np.pi*fx*xx + 2*np.pi*i*ft*params["dt"]).numpy() + 1) * 1/2
        vw.add(frame)

        # for assignment
        stack[i, :] = torch.ceil(0.75*peak_photons * torch.from_numpy(frame))+1  # round to whole numbers up so always have a photon

In [ ]:
# Show a fixed photon frame for comparison
plt.figure()
plt.imshow(torch.squeeze(stack[200,:]).detach().cpu().numpy())

In [ ]:
gain = 1 / np.log(15000)
dv = gain*np.log(stack[200,:]) - gain*np.log(stack[199,:])
print(dv.max(), dv.min())
dv[dv < params["neg_th"]] = -1
dv[dv >= params["pos_th"]] = 1
dv[(dv > params["neg_th"]) & (dv < params["pos_th"])] = 0
# Show an example log differential frame
plt.figure()
plt.imshow(dv)

print(stack[199].min())
print(gain*np.log(stack[199,:]).min())
print((gain*np.log(stack[200,:]) - gain*np.log(stack[199,:])).min())

## Passing Intensity Stack Through Event Simulator

In [ ]:
# Set requires gradient to true
stack.requires_grad_(True)

In [ ]:
# Now lets feed the video into the forward model and check the result
events, frames = es.forward(stack, reset=True)  # DO NOT directly call _events_from_stack as forward incorporates error catching protocols
# print(events) # Check if events are correct
# frames.backward(torch.ones_like(frames))

In [ ]:
# # Plot results
plt.figure()
plt.imshow(torch.squeeze(stack[200,:]).detach().cpu().numpy())

plt.figure()
plt.imshow(torch.squeeze(frames[200,:]).detach().cpu().numpy())

# plt.figure()
# plt.imshow(stack.grad[200,:])

# print(stack.grad.min())
# print(stack.grad.max())

# plt.figure()
# plt.imshow(torch.squeeze(frames[199,:]).detach().cpu().numpy())

In [ ]:
# visualize output frames to see event encoding

# write a video to the notebook to visualize result!
with VideoWriter(fps=30) as vw:
    for i in tqdm(range(vid_size[0])):

        # for visualization
        vw.add((frames[i,:].detach().cpu().numpy()+1)/2)

## Filter (Downstream Operations) Testing + Differentiability Verification/Gradient Examination

In [ ]:
frames = frames[:, None, :]
frames.retain_grad()
frames.shape

In [ ]:
HEIGHT = 50
WIDTH = 200

In [ ]:
ba_filter = BAFilter(time_threshold=1, height=HEIGHT, width=WIDTH)
ie_filter = IEFilter(thresh_negative=2, thresh_positive=3, height=HEIGHT, width=WIDTH)
ynoise_filter = YNoiseFilter(time_context_delta=2, space_window_size=3, density_threshold=4, height=HEIGHT, width=WIDTH)
filtered_frames = ba_filter.filter_frames_tensor(frames=frames, inplace=False, device='cuda', polarity_agnostic=False)
# filtered_frames = ynoise_filter.filter_frames_tensor(frames=frames, inplace=False, device='cuda')

In [ ]:
stack.retain_grad()
filtered_frames.backward(torch.ones_like(filtered_frames))

In [ ]:
frame_to_display = 200

plt.figure()
plt.imshow(stack[frame_to_display,:].detach().cpu().numpy())

plt.figure()
plt.imshow(torch.squeeze(frames[frame_to_display,:]).detach().cpu().numpy())

plt.figure()
plt.imshow(torch.squeeze(filtered_frames[frame_to_display,:]).detach().cpu().numpy())

plt.figure()
plt.imshow(stack.grad[frame_to_display,:].detach().cpu().numpy())

plt.figure()
plt.imshow(torch.squeeze(frames.grad[frame_to_display,:]).detach().cpu().numpy())

In [ ]:
frames.grad.mean()

In [ ]:
frames.grad.unique(), frames.grad.unique().size(0)

In [ ]:
stack.grad.unique(), stack.grad.unique().size(0)

In [ ]:
filtered_frames.unique()

In [ ]:
filtered_frames.nonzero().size(0) / frames.nonzero().size(0)

In [ ]:
torch.any(filtered_frames.to(frames.device) != frames)